# Protocole commun du benchmark AutoML

Ce notebook présente le protocole expérimental utilisé pour comparer plusieurs solutions AutoML dans le cadre de la prédiction de la volatilité du Bitcoin à 7 jours et à 30 jours.

Il permet de vérifier les datasets, de définir un découpage temporel sans fuite de données et de fixer des conditions d’évaluation aussi homogènes que possible. 

Les différentes solutions seront comparées à partir des mêmes variables, des mêmes périodes d’entraînement et de test, ainsi que des mêmes métriques, notamment la MAE et la RMSE.

L’objectif est de garantir une comparaison équitable, reproductible et suffisamment documentée pour sélectionner la solution AutoML la plus adaptée au projet VolatiChainXplorerAI.

In [37]:
# Charger les bibliothèques nécessaires
import sys
from pathlib import Path

import numpy as np
import pandas as pd


In [3]:
# Définir le chemin du projet
project_root = Path.cwd().parent
sys.path.insert(0,str(project_root))

# Définir le chemin vers les datasets
datasets_dir = (
    project_root
    / "data"
    / "exports"
    / "datasets"
    / "processed"
)

## Charger les deux datasets

In [12]:
path_7d = datasets_dir / "dataset_7d.csv"
path_30d = datasets_dir / "dataset_30d.csv"

if not path_7d.exists() or not path_30d.exists():
    raise FileNotFoundError(
        "Les datasets 7d et 30d sont introuvables"
    ) 
    
# Charger les datasets

dataset_7d = pd.read_csv(
    path_7d,
    parse_dates=["date"], 
    
)

dataset_30d = pd.read_csv(
    path_30d,
    parse_dates=["date"], 
    
)

display(dataset_7d.head(10))
display(dataset_30d.head(10))
print("Le dataset_7d à ", dataset_7d.shape)
print("Le dataset_30d à ", dataset_30d.shape)
print("Date 7 jours triée ?", dataset_7d['date'].is_monotonic_increasing)
print("Date 30 jours triée ?", dataset_30d['date'].is_monotonic_increasing)
print(
    "Dates dupliquées - 7 jours :",
    dataset_7d["date"].duplicated().sum()
)

print(
    "Dates dupliquées - 30 jours :",
    dataset_30d["date"].duplicated().sum()
)

,date,log_return,volatility_3d,volatility_7d,volatility_14d,volatility_30d,daily_range,volume_change,market_cap_change,rate_mro,inflation_rate,unemployment_rate,monetary_m3_rate,log_return_lag_1d,log_return_lag_7d,log_return_lag_30d,target_volatility_7d
0,2010-08-14,-0.001822,0.033331,0.038075,0.049320,0.081509,0.213845,0.222828,0.002811,1.0,1.6,10.1,0.868765,0.036224,-0.033055,0.008011,0.222463
1,2010-08-15,0.203901,0.109457,0.081983,0.060889,0.085857,0.312778,0.012217,0.228548,1.0,1.6,10.1,0.868765,-0.001822,0.019971,0.141146,0.205870
2,2010-08-16,0.203751,0.118731,0.102813,0.077270,0.093100,0.387127,1.393058,0.233132,1.0,1.6,10.1,0.868765,0.203901,0.083118,-0.009756,0.181932
3,2010-08-17,-0.445352,0.374803,0.217407,0.150031,0.120467,0.673585,0.232124,-0.358034,1.0,1.6,10.1,0.868765,0.203751,0.000105,0.177546,0.051001
4,2010-08-18,0.107709,0.350342,0.221537,0.152587,0.121868,0.272079,-0.776815,0.110958,1.0,1.6,10.1,0.868765,-0.445352,-0.017679,-0.059794,0.031894
5,2010-08-19,0.030756,0.299577,0.220862,0.152672,0.121940,0.210101,-0.750675,0.029959,1.0,1.6,10.1,0.868765,0.107709,-0.030205,-0.025075,0.030121
6,2010-08-20,-0.051178,0.079457,0.222213,0.153605,0.122015,0.194742,4.731556,-0.044544,1.0,1.6,10.1,0.868765,0.030756,0.036224,0.040740,0.028749
7,2010-08-21,0.037894,0.049494,0.222463,0.153331,0.113621,0.136476,1.503096,0.041647,1.0,1.6,10.1,0.868765,-0.051178,-0.001822,-0.241559,0.023984
8,2010-08-22,-0.008663,0.044552,0.205870,0.153417,0.113150,0.206362,0.793173,-0.006456,1.0,1.6,10.1,0.868765,0.037894,0.203901,0.061411,0.044033
9,2010-08-23,-0.011592,0.027764,0.181932,0.152056,0.111450,0.272261,-0.770218,-0.011956,1.0,1.6,10.1,0.868765,-0.008663,0.203751,0.107162,0.062035


,date,log_return,volatility_3d,volatility_7d,volatility_14d,volatility_30d,daily_range,volume_change,market_cap_change,rate_mro,inflation_rate,unemployment_rate,monetary_m3_rate,log_return_lag_1d,log_return_lag_7d,log_return_lag_30d,target_volatility_30d
0,2010-08-14,-0.001822,0.033331,0.038075,0.049320,0.081509,0.213845,0.222828,0.002811,1.0,1.6,10.1,0.868765,0.036224,-0.033055,0.008011,0.107934
1,2010-08-15,0.203901,0.109457,0.081983,0.060889,0.085857,0.312778,0.012217,0.228548,1.0,1.6,10.1,0.868765,-0.001822,0.019971,0.141146,0.100748
2,2010-08-16,0.203751,0.118731,0.102813,0.077270,0.093100,0.387127,1.393058,0.233132,1.0,1.6,10.1,0.868765,0.203901,0.083118,-0.009756,0.106682
3,2010-08-17,-0.445352,0.374803,0.217407,0.150031,0.120467,0.673585,0.232124,-0.358034,1.0,1.6,10.1,0.868765,0.203751,0.000105,0.177546,0.070091
4,2010-08-18,0.107709,0.350342,0.221537,0.152587,0.121868,0.272079,-0.776815,0.110958,1.0,1.6,10.1,0.868765,-0.445352,-0.017679,-0.059794,0.104685
5,2010-08-19,0.030756,0.299577,0.220862,0.152672,0.121940,0.210101,-0.750675,0.029959,1.0,1.6,10.1,0.868765,0.107709,-0.030205,-0.025075,0.104628
6,2010-08-20,-0.051178,0.079457,0.222213,0.153605,0.122015,0.194742,4.731556,-0.044544,1.0,1.6,10.1,0.868765,0.030756,0.036224,0.040740,0.104275
7,2010-08-21,0.037894,0.049494,0.222463,0.153331,0.113621,0.136476,1.503096,0.041647,1.0,1.6,10.1,0.868765,-0.051178,-0.001822,-0.241559,0.103997
8,2010-08-22,-0.008663,0.044552,0.205870,0.153417,0.113150,0.206362,0.793173,-0.006456,1.0,1.6,10.1,0.868765,0.037894,0.203901,0.061411,0.104015
9,2010-08-23,-0.011592,0.027764,0.181932,0.152056,0.111450,0.272261,-0.770218,-0.011956,1.0,1.6,10.1,0.868765,-0.008663,0.203751,0.107162,0.104048


Le dataset_7d à  (5579, 17)
Le dataset_30d à  (5533, 17)
Date 7 jours triée ? True
Date 30 jours triée ? True
Dates dupliquées - 7 jours : 0
Dates dupliquées - 30 jours : 0


## Vérifier les dimensions, dates, colonnes et cibles du dataset à 7 jours

In [5]:
print(dataset_7d.columns.tolist())
print(f"Nom du fichier : {path_7d.name}")
print(f"Nombre de lignes du fichier :{len(dataset_7d)}")
print(f"date de debut :{dataset_7d['date'].min()}")
print(f"date de fin :{dataset_7d['date'].max()}")
print(f"Nom de la cible :{dataset_7d['target_volatility_7d'].name}")
print(f"Nombre de variables explicatives :{len(dataset_7d.columns)-2}")

['date', 'log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d', 'target_volatility_7d']
Nom du fichier : dataset_7d.csv
Nombre de lignes du fichier :5579
date de debut :2010-08-14 00:00:00
date de fin :2026-09-05 00:00:00
Nom de la cible :target_volatility_7d
Nombre de variables explicatives :15


## vérifier les dimensions, dates, colonnes et cibles du dataset à 30 jours

In [6]:
print(dataset_30d.columns.tolist())
print(f"Nom du fichier : {path_30d.name}")
print(f"Nombre de lignes du fichier :{len(dataset_30d)}")
print(f"date de debut :{dataset_30d['date'].min()}")
print(f"date de fin :{dataset_30d['date'].max()}")
print(f"Nom de la cible :{dataset_30d['target_volatility_30d'].name}")
print(f"Nombre de variables explicatives :{len(dataset_30d.columns)-2}")

['date', 'log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d', 'target_volatility_30d']
Nom du fichier : dataset_30d.csv
Nombre de lignes du fichier :5533
date de debut :2010-08-14 00:00:00
date de fin :2026-08-13 00:00:00
Nom de la cible :target_volatility_30d
Nombre de variables explicatives :15


## Contrôler les doublons et valeurs manquantes

In [11]:

print("Nombre de doublons dans le dataset 7d :")
print(dataset_7d.duplicated().sum())
print("==================================")
print("Nombre de valeurs manquantes dans le dataset 7d :")
print(dataset_7d.isna().sum())
print("==================================")
print("Fin de l'analyse des doublons et des valeurs manquantes pour le dataset à 7d")

Nombre de doublons dans le dataset 7d :
0
Nombre de valeurs manquantes dans le dataset 7d :
date                    0
log_return              0
volatility_3d           0
volatility_7d           0
volatility_14d          0
volatility_30d          0
daily_range             0
volume_change           0
market_cap_change       0
rate_mro                0
inflation_rate          0
unemployment_rate       0
monetary_m3_rate        0
log_return_lag_1d       0
log_return_lag_7d       0
log_return_lag_30d      0
target_volatility_7d    0
dtype: int64
Fin de l'analyse des doublons et des valeurs manquantes pour le dataset à 7d


In [10]:
print("Nombre de doublons dans le dataset 30d :")
print(dataset_30d.duplicated().sum())
print("==================================")
print("Nombre de valeurs manquantes dans le dataset 30d :")
print(dataset_30d.isna().sum())
print("==================================")
print("Fin de l'analyse des doublons et des valeurs manquantes pour le dataset à 30d")

Nombre de doublons dans le dataset 30d :
0
Nombre de valeurs manquantes dans le dataset 30d :
date                     0
log_return               0
volatility_3d            0
volatility_7d            0
volatility_14d           0
volatility_30d           0
daily_range              0
volume_change            0
market_cap_change        0
rate_mro                 0
inflation_rate           0
unemployment_rate        0
monetary_m3_rate         0
log_return_lag_1d        0
log_return_lag_7d        0
log_return_lag_30d       0
target_volatility_30d    0
dtype: int64
Fin de l'analyse des doublons et des valeurs manquantes pour le dataset à 30d


## Définir le découpage temporel commun

In [20]:
def temporal_split_with_purge(
    dataset,
    horizon,
    validation_start="2023-01-01",
    test_start="2025-01-01",
    test_end="2026-08-13",
):
    """
    Découpe chronologiquement un dataset en trois parties
    et ajoute une zone de sécurité correspondant à l'horizon.
    """

    data = dataset.sort_values("date").copy()

    validation_start = pd.Timestamp(validation_start)
    test_start = pd.Timestamp(test_start)

    # Dernière date autorisée dans l'entraînement
    train_end = validation_start - pd.Timedelta(
        days=horizon + 1
    )

    # Dernière date autorisée dans la validation
    validation_end = test_start - pd.Timedelta(
        days=horizon + 1
    )
    
    # Dernière date autorisée dans le test
    test_end = pd.Timestamp(test_end)
    

    train = data.loc[
        data["date"] <= train_end
    ].copy()

    validation = data.loc[
        data["date"].between(
            validation_start,
            validation_end
        )
    ].copy()

    test = data.loc[
        data["date"].between(
            test_start,
            test_end
        )
    ].copy()

    return train, validation, test

In [23]:
# Découper le dataset à 7 jours
train_7d, validation_7d, test_7d = (
    temporal_split_with_purge(
        dataset=dataset_7d,
        horizon=7,
        test_end="2026-08-13"
    )
)

In [24]:
# Découper le dataset à 30 jours
train_30d, validation_30d, test_30d = (
    temporal_split_with_purge(
        dataset=dataset_30d,
        horizon=30
    )
)

In [26]:
# Contrôler les résultats
splits = {
    "train_7d": train_7d,
    "validation_7d": validation_7d,
    "test_7d": test_7d,
    "train_30d": train_30d,
    "validation_30d": validation_30d,
    "test_30d": test_30d
}

for name, data in splits.items():
    print(
        name,
        "| lignes :", len(data),
        "| début :", data["date"].min(),
        "| fin :", data["date"].max()
    )
    print("==============================")

train_7d | lignes : 4228 | début : 2010-08-14 00:00:00 | fin : 2022-12-24 00:00:00
validation_7d | lignes : 724 | début : 2023-01-01 00:00:00 | fin : 2024-12-24 00:00:00
test_7d | lignes : 590 | début : 2025-01-01 00:00:00 | fin : 2026-08-13 00:00:00
train_30d | lignes : 4182 | début : 2010-08-14 00:00:00 | fin : 2022-12-01 00:00:00
validation_30d | lignes : 701 | début : 2023-01-01 00:00:00 | fin : 2024-12-01 00:00:00
test_30d | lignes : 590 | début : 2025-01-01 00:00:00 | fin : 2026-08-13 00:00:00


In [30]:
assert train_7d["date"].max() < validation_7d["date"].min()
assert validation_7d["date"].max() < test_7d["date"].min()

assert train_30d["date"].max() < validation_30d["date"].min()
assert validation_30d["date"].max() < test_30d["date"].min()

assert test_7d["date"].reset_index(drop=True).equals(
    test_30d["date"].reset_index(drop=True)
)

print("ok - Découpage temporel validé : aucune superposition.")
print("ok - Les deux tests utilisent exactement les mêmes dates.")

ok - Découpage temporel validé : aucune superposition.
ok - Les deux tests utilisent exactement les mêmes dates.


## Fixer les variables explicatives communes

In [29]:
features_7d = [
    column for column in dataset_7d.columns
    if column not in ["date", "target_volatility_7d"]
]

features_30d = [
    column for column in dataset_30d.columns
    if column not in ["date", "target_volatility_30d"]
]

print("Variables 7 jours :", features_7d)
print("Variables 30 jours :", features_30d)

assert features_7d == features_30d

feature_columns = features_7d

print("ok - Variables explicatives identiques.")
print("Nombre de variables :", len(feature_columns))

Variables 7 jours : ['log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d']
Variables 30 jours : ['log_return', 'volatility_3d', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'daily_range', 'volume_change', 'market_cap_change', 'rate_mro', 'inflation_rate', 'unemployment_rate', 'monetary_m3_rate', 'log_return_lag_1d', 'log_return_lag_7d', 'log_return_lag_30d']
ok - Variables explicatives identiques.
Nombre de variables : 15


### Séparation du dataset à 7 jours en sous ensmbles train, validation, test et target

In [ ]:

X_train_7d = train_7d[
    feature_columns
].reset_index(drop=True)

y_train_7d = train_7d[
    "target_volatility_7d"
].reset_index(drop=True)

X_validation_7d = validation_7d[
    feature_columns
].reset_index(drop=True)

y_validation_7d = validation_7d[
    "target_volatility_7d"
].reset_index(drop=True)

X_test_7d = test_7d[
    feature_columns
].reset_index(drop=True)

y_test_7d = test_7d[
    "target_volatility_7d"
].reset_index(drop=True)

dates_test_7d = test_7d["date"].reset_index(drop=True)

In [ ]:
# Vérification des dimensions des ensembles d'entraînement, de validation et de test à 7 jours
print("X_train_7d :", X_train_7d.shape)
print("y_train_7d :", y_train_7d.shape)

print("X_validation_7d :", X_validation_7d.shape)
print("y_validation_7d :", y_validation_7d.shape)

print("X_test_7d :", X_test_7d.shape)
print("y_test_7d :", y_test_7d.shape)

assert len(X_train_7d) == len(y_train_7d)
assert len(X_validation_7d) == len(y_validation_7d)
assert len(X_test_7d) == len(y_test_7d)
assert X_train_7d.shape[1] == 15

print("ok - Séparation X/y à 7 jours validée.")

X_train_7d : (4228, 15)
y_train_7d : (4228,)
X_validation_7d : (724, 15)
y_validation_7d : (724,)
X_test_7d : (590, 15)
y_test_7d : (590,)
ok - Séparation X/y à 7 jours validée.


### Séparation du dataset à 30 jours en sous ensmbles train, validation, test et target

In [33]:
X_train_30d = train_30d[
    features_30d
].reset_index(drop=True)

y_train_30d = train_30d[
    "target_volatility_30d"
].reset_index(drop=True)

X_validation_30d = validation_30d[
    features_30d
].reset_index(drop=True)

y_validation_30d = validation_30d[
    "target_volatility_30d"
].reset_index(drop=True)

X_test_30d = test_30d[
    features_30d
].reset_index(drop=True)

y_test_30d = test_30d[
    "target_volatility_30d"
].reset_index(drop=True)

dates_test_30d = test_30d["date"].reset_index(drop=True)

In [34]:
# Vérification des dimensions des ensembles d'entraînement, de validation et de test à 7 jours
print("X_train_30d :", X_train_30d.shape)
print("y_train_30d :", y_train_30d.shape)

print("X_validation_30d :", X_validation_30d.shape)
print("y_validation_30d :", y_validation_30d.shape)

print("X_test_30d :", X_test_30d.shape)
print("y_test_30d :", y_test_30d.shape)

assert len(X_train_30d) == len(y_train_30d)
assert len(X_validation_30d) == len(y_validation_30d)
assert len(X_test_30d) == len(y_test_30d)
assert X_train_30d.shape[1] == 15

print("ok - Séparation X/y à 30 jours validée.")

X_train_30d : (4182, 15)
y_train_30d : (4182,)
X_validation_30d : (701, 15)
y_validation_30d : (701,)
X_test_30d : (590, 15)
y_test_30d : (590,)
ok - Séparation X/y à 30 jours validée.


## Vérifier les types et les valeurs infinies

Les solutions AutoML doivent recevoir uniquement des valeurs numériques finies. Une valeur inf peut exister même lorsque le contrôle des valeurs manquantes indique zéro.

In [39]:
X_sets= {
    "X_train_7d" : X_train_7d,    
    "X_validation_7d" : X_validation_7d,    
    "X_test_7d" : X_test_7d,    
    "X_train_30d" : X_train_30d,    
    "X_validation_30d" : X_validation_30d,    
    "X_test_30d" : X_test_30d,
}

y_sets = {
    "y_train_7d" : y_train_7d,
    "y_validation_7d" : y_validation_7d,
    "y_test_7d" : y_test_7d,
    "y_train_30d" : y_train_30d,
    "y_validation_30d" : y_validation_30d,
    "y_test_30d" : y_test_30d,
}

for name, data in X_sets.items():
    non_numeric = data.select_dtypes(exclude="number").columns.tolist()
    finite = np.isfinite(data.to_numpy()).all()
    
    print(
        name,
        "| Colonne non numeriques :", non_numeric,
        "| Valeurs finies :", finite
    )
for name, target in y_sets.items():
    finite = np.isfinite(target.to_numpy()).all()
    print(
        name,
        "| Type :", target.dtypes,
        "| Valeurs finies :", finite,
    )

X_train_7d | Colonne non numeriques : [] | Valeurs finies : True
X_validation_7d | Colonne non numeriques : [] | Valeurs finies : True
X_test_7d | Colonne non numeriques : [] | Valeurs finies : True
X_train_30d | Colonne non numeriques : [] | Valeurs finies : True
X_validation_30d | Colonne non numeriques : [] | Valeurs finies : True
X_test_30d | Colonne non numeriques : [] | Valeurs finies : True
y_train_7d | Type : float64 | Valeurs finies : True
y_validation_7d | Type : float64 | Valeurs finies : True
y_test_7d | Type : float64 | Valeurs finies : True
y_train_30d | Type : float64 | Valeurs finies : True
y_validation_30d | Type : float64 | Valeurs finies : True
y_test_30d | Type : float64 | Valeurs finies : True


## Fixer le calcul commun des métriques

Les métriques sont calculées sur les prédictions du test afin de ne pas reprendre les résultats affichés par chaque outil.
- MAE : erreur moyenne entre la volatilité prévue et la volatilité réelle.
- RMSE : erreur qui pénalise plus fortement les grosses erreurs.
- Pour les deux métriques, **plus la valeur est faible, meilleur est le modèle**.

Afin d'obtenir un benchmark** équitable**, toutes les solutions sont évaluées :
- sur les mêmes 590 observations ;
- avec les mêmes valeurs réelles ;
- avec les mêmes formules MAE et RMSE.

In [41]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.3 MB 4.2 MB/s eta 0:00:02
   -------- ------------------------------- 1.8/8.3 MB 4.6 MB/s eta 0:00:02
   ---------- ----------------------------- 2.1/8.3 MB 4.5 MB/s eta 0:00:02
   ------------- -------------------------- 2.9/8.3 MB 3.2 MB/s eta 0:00:02
   ----------------- ---------------------- 3.7/8.3 MB 3.3 MB/s eta 0:00:02
   --------------------- ------------------ 4.5/8.3 MB 3.4 MB/s eta 0:00:02
   ------------------------- -------------- 5.2/8.3 MB 3.5 MB/s eta 0:00:01
   ----------------------------- ---------- 6.0/8.3 MB 3.5 MB/s eta 0:00:01
   ---------------------------------- ----- 7.1/8.3 MB 3.6 MB/s eta 0:00:01
   -------------------------------------- - 7.9/8.3 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 3.6 MB/s  0:00:02

   ---------- ----------------------------- 1/4 [cloudpickle]
   -------------------- -----------

In [42]:
import sklearn

print(sklearn.__version__)

1.9.1


In [44]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


def calculate_regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )   
    return {
        "MAE": mae,
        "RMSE": rmse
    }

## Fixer les conditions d’exécution 
Pour comparer équitablement les solutions, nous devons leur imposer notamment :
- la même durée maximale d’entraînement ;
- la même graine aléatoire ;
- le même ordinateur ;
- le CPU uniquement, sauf décision contraire.

In [46]:
RANDOM_SEED = 42
TRAINING_TIME_LIMIT_SECONDS = 600
TRAINING_TIME_LIMIT_MINUTES = 10
COMPUTE_DEVICE = "CPU"

## Mesurer les temps de manière commune
Nous utiliserons time.perf_counter(), qui mesure précisément la durée d’une opération.

Cette fonction  servira à mesurer pour chaque solution:
- le temps d’entraînement ;
- le temps d’inférence.

In [48]:
import time


def measure_execution_time(function, *args, **kwargs):
    """
    Exécute une fonction et mesure sa durée en secondes.
    """
    
    start_time = time.perf_counter()
    result = function(*args, **kwargs)
    end_time = time.perf_counter()
    elapsed_time = end_time - start_time
    
    return result, elapsed_time

## Figer les six échantillons

- Nous enregistrons les découpages sous forme de fichiers CSV définitifs.
- Chaque POC relira ces fichiers au lieu de reconstruire son propre découpage.
- Cela garantit que PyCaret, AutoGluon et H2O utilisent exactement les mêmes lignes.

In [55]:
benchmark_dir = path_7d.parents[1] / "benchmark_splits"
benchmark_dir.mkdir(parents=True, exist_ok=True)

splits_to_export = {
    "train_7d.csv" : train_7d,
    "validation_7d.csv" : validation_7d,
    "test_7d.csv" : test_7d,
    "train_30d.csv" : train_30d,
    "validation_30d.csv" : validation_30d,
    "test_30d.csv" : test_30d,
}

for filename, data in splits_to_export.items():
    output = benchmark_dir / filename
    data.to_csv(
        output,
        index=False
    )
    print(
        filename,
        "| Nombre de lignes :", len(data),
        "| enregistré : ", output.exists()
    )


train_7d.csv | Nombre de lignes : 4228 | enregistré :  True
validation_7d.csv | Nombre de lignes : 724 | enregistré :  True
test_7d.csv | Nombre de lignes : 590 | enregistré :  True
train_30d.csv | Nombre de lignes : 4182 | enregistré :  True
validation_30d.csv | Nombre de lignes : 701 | enregistré :  True
test_30d.csv | Nombre de lignes : 590 | enregistré :  True


In [56]:
# Vérification 
expected_rows = {
    "train_7d.csv": 4228,
    "validation_7d.csv": 724,
    "test_7d.csv": 590,
    "train_30d.csv": 4182,
    "validation_30d.csv": 701,
    "test_30d.csv": 590
}


for filename, expected_count in expected_rows.items():
    file_path = benchmark_dir / filename
    verification = pd.read_csv(file_path)

    assert file_path.exists()
    assert len(verification) == expected_count
    assert verification.shape[1] == 17

    print(
        filename,
        "| dimensions :", verification.shape,
        "| validation : OK"
    )

train_7d.csv | dimensions : (4228, 17) | validation : OK
validation_7d.csv | dimensions : (724, 17) | validation : OK
test_7d.csv | dimensions : (590, 17) | validation : OK
train_30d.csv | dimensions : (4182, 17) | validation : OK
validation_30d.csv | dimensions : (701, 17) | validation : OK
test_30d.csv | dimensions : (590, 17) | validation : OK


## Fixer l’utilisation des trois ensembles

### Règle d’utilisation des données

- **Entraînement** : utilisé pour faire apprendre les modèles.
- **Validation** : utilisée pour comparer les modèles et sélectionner le meilleur candidat.
- **Test** : utilisé une seule fois pour mesurer la performance finale du modèle sélectionné.

- Le **test** ne doit jamais intervenir dans l’apprentissage, le réglage des paramètres ou le choix du modèle.
- Pour chaque **horizon**, toutes les solutions AutoML utilisent les mêmes fichiers d’entraînement, de validation et de test.
- La **colonne date** est conservée pour les contrôles, mais retirée des variables données au modèle.
- Les différences de fonctionnement interne propres à chaque solution AutoML seront documentées.

Pendant les POC, chaque solution AutoML sera entraînée uniquement sur l’ensemble d’entraînement, puis comparée sur l’ensemble de validation. Le jeu de test restera totalement inutilisé à cette étape. Après la sélection de la meilleure solution et du meilleur modèle pour chaque horizon, le modèle retenu sera réentraîné sur les données d’entraînement et de validation réunies, puis évalué une seule fois sur le jeu de test.

L’ordre méthodologique est :
1. entraînement des candidats sur train ;
2. comparaison des candidats sur validation avec MAE et RMSE ;
3. sélection de la solution et du modèle ;
4. réentraînement du modèle retenu sur train + validation ;
5. évaluation finale unique sur test.

## Fixer la mesure des ressources


Pour chaque entraînement, nous mesurerons :
- le temps total en secondes ;
- le pic d’utilisation du CPU en pourcentage ;
- le pic de mémoire vive utilisé en mégaoctets ;
- le temps d’inférence sur l’ensemble concerné.

La consommation des ressources sera mesurée avec psutil pendant chaque expérimentation. Le CPU et la mémoire vive seront relevés à intervalles réguliers afin d’identifier leurs valeurs maximales. Les tests seront exécutés sur le même ordinateur, avec les applications inutiles fermées, afin de limiter les perturbations extérieures.

In [57]:
%pip install psutil

Note: you may need to restart the kernel to use updated packages.


In [58]:
import psutil

print("Version de psutil :", psutil.__version__)

Version de psutil : 7.0.0


## Préparer le tableau de résultats

Signification des principales colonnes :

- solution : PyCaret, AutoGluon ou H2O ;
- horizon_days : 7 ou 30 ;
- model : modèle sélectionné par la solution ;
- evaluation_set : validation pendant le benchmark, puis test pour le modèle final ;
- mae et rmse : performances prédictives ;
- training_time_seconds : durée totale d’entraînement ;
- inference_time_seconds : durée totale des prédictions ;
- peak_cpu_percent et peak_ram_mb : ressources maximales utilisées ;
- estimated_cost_eur : coût éventuel ;
- notes : difficultés d’installation, d’intégration ou d’utilisation.

In [59]:
benchmark_columns = [
    "solution",
    "horizon_days",
    "model",
    "evaluation_set",
    "mae",
    "rmse",
    "training_time_seconds",
    "inference_time_seconds",
    "inference_time_per_row_ms",
    "peak_cpu_percent",
    "peak_ram_mb",
    "estimated_cost_eur",
    "notes"
]


benchmark_results = pd.DataFrame(
    columns=benchmark_columns
)

benchmark_results

,solution,horizon_days,model,evaluation_set,mae,rmse,training_time_seconds,inference_time_seconds,inference_time_per_row_ms,peak_cpu_percent,peak_ram_mb,estimated_cost_eur,notes


In [64]:
project_root = Path("__file__").resolve().parent.parent
print("Project root :", project_root)
path_7d.parents[2]
print("Path 7d parents[2] :", path_7d.parents[2])

Project root : D:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI
Path 7d parents[2] : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\data\exports


In [66]:

results_dir = path_7d.parents[2] / "results_benchmark" 
results_dir.mkdir(parents=True, exist_ok=True)

results_path = results_dir / "benchmark_results.csv"

benchmark_results.to_csv(
    results_path,
    index=False
)

print(
    "Tableau créé :",
    results_path.exists()
)

Tableau créé : True
